In [1]:
# Load environment and suppress warnings

import os
import warnings
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

if os.getenv("OPENAI_API_KEY"):
    model = ChatOpenAI(model="gpt-5-mini", temperature=0)
else:
    model = None
    print("Skipping model init. No API key found.")


In [3]:
# Summarize ticket

summary_prompt = PromptTemplate.from_template(
"""
Summarize the following support ticket in one concise sentence.

Ticket:
{ticket_text}
"""
)

summary_chain = summary_prompt | model | StrOutputParser()

In [4]:
# Step 2: Assign priority based on summary

priority_prompt = PromptTemplate.from_template(
"""
Based on the summary below, classify priority as:
High, Medium, or Low.

Return only the priority level.

Summary:
{summary}
"""
)

priority_chain = priority_prompt | model | StrOutputParser()

In [5]:
# Step 3: Recommend next action

action_prompt = PromptTemplate.from_template(
"""
Given the ticket summary and priority, recommend the next action in one sentence.

Summary:
{summary}

Priority:
{priority}
"""
)

action_chain = action_prompt | model | StrOutputParser()

In [6]:
# Multi-step execution with visible intermediate outputs

ticket_text = "Customer reports system outage. All users are unable to access the application."

# Step 1
summary = summary_chain.invoke({"ticket_text": ticket_text})
print("=== STEP 1: SUMMARY ===")
print(summary)
print()

# Step 2
priority = priority_chain.invoke({"summary": summary})
print("=== STEP 2: PRIORITY ===")
print(priority)
print()

# Step 3
action = action_chain.invoke({
    "summary": summary,
    "priority": priority
})

print("=== STEP 3: RECOMMENDED ACTION ===")
print(action)

=== STEP 1: SUMMARY ===
Customer reports a system-wide outage preventing all users from accessing the application.

=== STEP 2: PRIORITY ===
High

=== STEP 3: RECOMMENDED ACTION ===
Immediately declare a major incident and page the on-call SRE/operations team to begin the outage runbook—confirm scope via monitoring and logs, attempt failover/rollback if available, and notify stakeholders and customers.


In [7]:
# Execute the same multi-step workflow with a lower-priority ticket

ticket_text_2 = "Customer is requesting a minor update to their profile display name."

# Step 1
summary_2 = summary_chain.invoke({"ticket_text": ticket_text_2})
print("=== STEP 1 (TICKET 2): SUMMARY ===")
print(summary_2)
print()

# Step 2
priority_2 = priority_chain.invoke({"summary": summary_2})
print("=== STEP 2 (TICKET 2): PRIORITY ===")
print(priority_2)
print()

# Step 3
action_2 = action_chain.invoke({
    "summary": summary_2,
    "priority": priority_2
})

print("=== STEP 3 (TICKET 2): RECOMMENDED ACTION ===")
print(action_2)

=== STEP 1 (TICKET 2): SUMMARY ===
Customer requests a minor update to their profile display name.

=== STEP 2 (TICKET 2): PRIORITY ===
Low

=== STEP 3 (TICKET 2): RECOMMENDED ACTION ===
Ask the customer to confirm the exact new display name (and verify account ownership if required), then make the change during regular low-priority processing and notify them when it’s done.
